# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    worm_step,
    run_worm
)
import jax
import jax.numpy as jnp
import numpy as np
from typing import Tuple, Dict
from jax import Array
from jax.typing import ArrayLike
from functools import partial

In [2]:
length = 11  # 5
width = 11  # 5
p = 3
d = 2 * p
moebius_code = MoebiusCodeTwoOddPrime(length=length, width=width, d=d)
h_z = moebius_code.h_z
h_x = moebius_code.h_x
h_z_mod_2 = moebius_code.h_z_mod_2
h_z_mod_p = moebius_code.h_z_mod_p
h_x_mod_2 = moebius_code.h_x_mod_2
h_x_mod_p = moebius_code.h_x_mod_p
logical_x = moebius_code.logical_x
logical_z = moebius_code.logical_z
num_plaquette, num_edges = h_x.shape
gamma_t = 0.3
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=d, gamma_t=gamma_t
)

# error_master_key = jax.random.PRNGKey(42)
# initial_error = em_lindblad.generate_random_error(error_key)
# initial_error_mod_2 = jnp.mod(initial_error, 2)
# initial_error_mod_p = jnp.mod(initial_error, p)
# # Here we consider the full syndrome including the plaquette
# # we usually remove because of the constraint as this simplified the
# # coding of the worm algorithm. In fact, in this the syndromes will
# # always be annihilated in pairs, and the total number of syndromes is
# # always even as one can check numerically.
# syndrome = moebius_code.get_plaquette_syndrome(initial_error)
# syndrome_mod_2 = jnp.mod(syndrome, 2)
# syndrome_mod_p = jnp.mod(syndrome, p)
# num_plaquette, num_edges = h_x.shape

In [3]:
master_worm_key = jax.random.PRNGKey(144)
num_worms = 100
worm_keys = jax.random.split(master_worm_key, num_worms)

error_master_key = jax.random.PRNGKey(42)
num_samples = 100
error_keys = jax.random.split(master_worm_key, num_worms)


def generate_initial_worm_errors(
    key: ArrayLike,
    error_model: ErrorModelLindbladTwoOddPrime
) -> ArrayLike:
    initial_error = error_model.generate_random_error(key)
    initial_error_mod_2 = jnp.mod(initial_error, 2)
    initial_error_mod_p = jnp.mod(initial_error, p)
    initial_worm_error = jnp.vstack((initial_error_mod_2, initial_error_mod_p))
    return initial_worm_error


initial_worm_errors = jax.vmap(
    generate_initial_worm_errors, in_axes=(0, None))(error_keys, em_lindblad)

In [5]:
# master_worm_key = jax.random.PRNGKey(144)
# num_worms = 100
# worm_keys = jax.random.split(master_worm_key, num_worms)

# error_master_key = jax.random.PRNGKey(42)
# num_samples = 100
# worm_keys = jax.random.split(master_worm_key, num_worms)
# initial_error = em_lindblad.generate_random_error(error_key)

max_steps = 1000
initial_worm_state = {}
# worm_error = jnp.vstack(
#     (initial_error_mod_2, initial_error_mod_p))
initial_head = 18
initial_worm_state["head"] = initial_head
initial_worm_state["tail"] = initial_head
initial_worm_state["worm_success"] = False
initial_worm_state["accepted_moves"] = 0
initial_worm_state["attempted_moves"] = 0

run_worm_partial = partial(
    run_worm,
    initial_worm_state=initial_worm_state,
    h_error_mod_p=h_z_mod_p,
    h_mod_p=h_x_mod_p,
    error_model=em_lindblad,
    max_worm_steps=max_steps
)

# First over keys
run_worm_vmap = jax.vmap(run_worm_partial, in_axes=(None, 0))
# Then over initial errors
run_worm_vmap = jax.vmap(run_worm_vmap, in_axes=(0, None))
run_worm_jit = jax.jit(run_worm_vmap)
new_worm_state = run_worm_jit(initial_worm_errors, worm_keys)

# index = 0
# print("Number of accepted moves: {}".format(new_worm_state["accepted_moves"][index]))
# print("Number of attempted moves: {}".format(
#     new_worm_state["attempted_moves"][index]))

In [6]:
index = 28
print("Number of accepted moves: {}".format(
    new_worm_state["accepted_moves"][index]))
print("Number of attempted moves: {}".format(
    new_worm_state["attempted_moves"][index]))
print("Success: {}".format(
    new_worm_state["worm_success"][index]))

Number of accepted moves: [10 57  2 49  2  2 14 22  2 42 12  4 38 42 56 68 39 54 48  6 69  2  2 26
 63  4 73  2 10  2 58  4 36 54 41  2  4 20  2 48 50  4 28  8 42 24 10  6
 38 45 24 53  6 38  6 16  4  4 32 10 59 60 26  2 28 71  2 24  2  2  6 16
 14 36 63  2 26 32 53 30 23  6 35  2 18  2  2 10 68 43 10 70  2  2 53 16
  6 24 38 24]
Number of attempted moves: [ 221 1000   10 1000    5   10  128  387   36  795  102   12  706  574
 1000 1000 1000 1000  962  205 1000   21    7  524 1000   61 1000    2
  289   52 1000   29  480  954 1000   32   42  384   17  791  968  144
  698  210  772  476   77  157 1000 1000  550 1000   74 1000  109  214
   14   13  891   84 1000  793  303   22  487 1000   25  715    6   22
   32  295  262 1000 1000   29  621  276 1000  314 1000  127 1000   15
  194    5    9   65 1000 1000  182  687   16    3 1000  195  277  594
  985  405]
Success: [ True False  True False  True  True  True  True  True  True  True  True
  True  True False False False False  True  True F

In [9]:
print(new_worm_state["head"])
print(initial_worm_state["head"])

[[18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 ...
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]
 [18 18 18 ... 18 18 18]]
18


In [15]:
initial_worm_errors[0][0]

Array([0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0], dtype=int32)

In [16]:
index_a = 0
index_b = 0
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][index_a, index_b][0, :] - initial_worm_errors[index_a][0, :], 2))))

1


In [17]:
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][index_a, index_b][1, :] - initial_worm_errors[index_a][1, :], p))))

2


In [ ]:
syndrome_mod_2 = jnp.mod(h_x_mod_2 @ initial_worm_errors[index_a][0, :], 2)
new_syndrome_mod_2 = jnp.mod(
    h_x_mod_2 @ new_worm_state["worm_error"][index_a, index_b][0, :], 2)

jnp.mod(new_syndrome_mod_2 - syndrome_mod_2, 2)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [ ]:
syndrome_mod_p = jnp.mod(h_x_mod_p @ initial_worm_errors[index_a][1, :], p)
new_syndrome_mod_p = jnp.mod(
    h_x_mod_p @ new_worm_state["worm_error"][index_a, index_b][1, :], p)

jnp.mod(new_syndrome_mod_p - syndrome_mod_p, p)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)